### IMPORTAMOS LAS LIBRERÍAS NECESARIAS

In [9]:
import random
import time
import numpy as np
from IPython.display import clear_output
import sys

### DEFINICIÓN DE LA CLASE PARA JUGAR

In [47]:
class hexapawn:
    def __init__(self, seed=27779):
        self.seed = int(seed)
        self.gen = random.Random(self.seed)

    #Regresa un número entero aleatorio
    def random_int(self):
        return self.gen.randint(1, 1000)
    
    #Coloca el tablero para un juego desde 0
    def poner_el_juego(self, n, turno=0):
        primera_fila=[2 for i in range(n)]
        ultima_fila=[1 for i in range(n)]
        fila_medio=[0 for i in range(n)]

        tablero=[]
        tablero.append(primera_fila.copy())
        for i in range(n-2): tablero.append(fila_medio.copy())
        tablero.append(ultima_fila.copy())

        return tablero, 1 if turno==0 or turno==1 else 2
    
    #Imprime de manera bonita el estado del juego, junto con el turno
    def ver_tablero(self, tablero, turno=1):
        #clear_output()
        count=0
        print("\n╔", end="")
        for i in range(len(tablero)*4-1):
            print("═", end="")
            count+=1
            if(count==100):
                return 
            
        print("╗   filas     turno de: ", "X" if turno==1 else "O")
        
        for i in range(len(tablero)):
            print("║", end="")

            for j in range(len(tablero[1])):
                print("   " if tablero[i][j]==0 else " X " if tablero[i][j]==1 else " O ", end="")
                if j!=len(tablero[1])-1: print("│", end="")
                
            print("║ ", i)
            
            if i!=len(tablero)-1:
                print("║", end="")
                for j in range(len(tablero[1])):
                    for k in range(3):
                        print("-", end="")
                        
                    if j!=len(tablero[1])-1:
                        print("┼", end="")
                print("║")
        
        print("╚", end="")
        for i in range(len(tablero[1])*4-1):
            print("═", end="")

        print("╝\n ", end="")
        for i in range(len(tablero[1])):
            print(f" {i}  ", end="")
            
        print("\n\ncolumnas\n\n")
        
    #Cambia de turno para el siguiente
    def siguiente_turno(self, turno):
        return 2 if turno==2 else 1
    
    #Filas original, columnas original, filas terminal, columnas terminal
    #te dice si el tiro que quieres hacer es legal o no
    def tiro_legal(self, f_o, c_o, f_t, c_t, tablero):
        if(f_t<0 or f_t>len(tablero)-1 or c_t<0 or c_t>len(tablero)-1):
            return False
        
        if(tablero[f_o][c_o]==1):
            return (((f_t==f_o-1) and (c_o==c_t)) and (tablero[f_t][c_t]==0)) or (((f_t==f_o-1) and (c_o==c_t-1)) and (tablero[f_t][c_t]==2)) or (((f_t==f_o-1) and (c_o==c_t+1)) and (tablero[f_t][c_t]==2))
        
        elif(tablero[f_o][c_o]==2):
            return (((f_t==f_o+1) and (c_o==c_t)) and (tablero[f_t][c_t]==0)) or (((f_t==f_o+1) and (c_o==c_t-1)) and (tablero[f_t][c_t]==1)) or (((f_t==f_o+1) and (c_o==c_t+1)) and (tablero[f_t][c_t]==1));
        
        else:
            return False
    
    #recibe un tablero junto con el turno y da un tiro aleatorio, regresa el tablero con el
    #tiro regristrado y el siguiente turno en una tupla
    def tiro_random(self, tablero, turno):
        #posibles almacena un vector, de coordenadas de todas las fichas del jugador del que queremos hacer el tiro
        posibles=[]

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]==turno):
                    posibles.append([i, j])

        self.gen.shuffle(posibles)
        
        while posibles:
            direcciones=[-1,0,1]
            elegido=posibles[len(posibles)-1]
            self.gen.shuffle(direcciones)
            
            while direcciones:
                if(self.tiro_legal(elegido[0],
                                   elegido[1],
                                   elegido[0]+(-1 if turno==1 else 1),
                                   elegido[1]+direcciones[len(direcciones)-1],
                                   tablero)):
                    tablero[elegido[0]][elegido[1]]=0
                    tablero[elegido[0]-1 if turno==1 else elegido[0]+1][elegido[1]+direcciones[len(direcciones)-1]]=turno

                    return (tablero, 2 if turno==1 else 1)
                
                else:
                    direcciones.pop()
                
            
            posibles.pop()
        
        return tablero, 0
    
    def todos_los_tiros(self, tablero, turno):
        posibles=[]

        todos=[]
        tablero_copia=tablero.copy()

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]==turno):
                    posibles.append([i, j])

        self.gen.shuffle(posibles)

        while(posibles):
            direcciones=[-1,0,1]
            elegido=posibles[len(posibles)-1]
            self.gen.shuffle(direcciones)
            
            while(direcciones):
                if(self.tiro_legal(elegido[0],
                                   elegido[1],
                                   elegido[0]+(-1 if turno==1 else 1),
                                   elegido[1]+direcciones[len(direcciones)-1],
                                   tablero)):
                    
                    tablero[elegido[0]][elegido[1]]=0
                    
                    tablero[elegido[0]-1 if turno==1 else elegido[0]+1][elegido[1]+direcciones[len(direcciones)-1]]=turno
                    todos.append(tablero)
                    tablero=tablero_copia
                    direcciones.pop()
                
                else:
                    direcciones.pop()
                
            
            posibles.pop()

        return todos, turno

    #condicion 1 de ganar: llegar al otro lado, regresa -1 si nadie satisface esta 
    #condición, de lo contrario, regresa la ficha ganadora
    def ganar_1(self, tablero):
        for i in range(0, len(tablero), len(tablero)-1):
            for j in range(len(tablero[0])):
                if i==0:
                    if(tablero[i][j]==1):
                        return True
                else:
                    if(tablero[i][j]==2):
                        return True
        return False
    
    #condicion 2 de ganar: uno de los dos ya no tiene fichas, regresa -1 si nadie satisface, 
    #de lo contrario regresa la ficha ganadora
    
    #creo que aquí hay un problema, no siempre detecta cuando ya no pueden seguir los tiros
    #al menos creo que esa es la razón de un problema que surgió al generar los tiros aleatorios
    #que para terminar los tiros pide un tablero con finalizacion, pero no se detectó por parte de esta funcion
    #pero ese fue un solo caso particular, en los demás si sigue funcionando
    def ganar_2(self, tablero):
        conteo=[0, 0]
        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]!=0):
                    conteo[tablero[i][j]-1]+=1
                    
        return conteo[0]==0 or conteo[1]==0

    #condicion 3 de ganar: ya nadie puede tirar. regresa 1 si esta condicion se cumple,
    #regresa 0 de lo contrario
    def ganar_3(self, tablero):
        fichas=[]

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j])!=0:
                    fichas.append([i, j])

        for i in range(len(fichas)):
            direcciones=[-1, 0, 1]
            for j in range(3):
                if(self.tiro_legal(fichas[i][0],
                                   fichas[i][1],
                                   fichas[i][0]+(-1 if tablero[fichas[i][0]][fichas[i][1]]==1 else 1),
                                   fichas[i][1]+direcciones[j],
                                   tablero)):
                    return False

        return True
    
    #verifica las condiciones de terminado del juego
    def condiciones_de_terminado(self, tablero):
        return self.ganar_1(tablero) or self.ganar_2(tablero) or self.ganar_3(tablero)
    
    #le das un tablero y turno, y termina el juego con tiros aleatorios. Regresa 
    #la ficha ganadora. Adicionalmente, puedes ver qué onda con el juego
    def terminar_juego(self, tablero, turno, verbose=0):
        n=0
        while(1):
            tablero, turno=self.tiro_random(tablero, turno)
            if(verbose):
                self.ver_tablero(tablero,turno)

            if(self.condiciones_de_terminado(tablero) or turno==0):
                if(verbose):
                    if(turno==1):
                        print("VICTORIA DE O")
                    else:
                        print("VICTORIA DE O")
                        
                return 2 if turno==1 else 1
            
        return 0
    
    #te da un juego contra humano dado un tamaño de tablero, no regresa nada, solo es
    #para entretenimiento
    def juego_random_contra_humano(self, tamano):
        print("JUEGO CONTRA HUMANO:")
        tablero, turno=self.poner_el_juego(tamano)
        
        seguir=1
        self.ver_tablero(tablero,turno)

        while(seguir==1):
            while(1):
                f_o=int(input("\nFilas(origen):"))
                c_o=int(input("\nColumnas(origen):"))
                f_t=int(input("\nFilas(terminal):"))
                c_t=int(input("\nColumnas(terminal):"))
                if(self.tiro_legal(f_o, 
                                   c_o,
                                   f_t,
                                   c_t,
                                   tablero)==True):
                    tablero[f_o][c_o]=0
                    tablero[f_t][c_t]=1
                    break
                else:
                    print("\nTiro no permitido, intenta de nuevo.")

            if(turno==1):
                turno=2
            else:
                turno=1

            self.ver_tablero(tablero,turno)

            if(self.condiciones_de_terminado(tablero)):
                print("\nVICTORIA DE:", "O\n\n\nQuieres seguir?:(1=si, 0=no)" if turno==1 else "X\n\n\nQuieres seguir?:(1=si, 0=no)")
                seguir=int(input())
                if(seguir!=0):
                    tablero, turno=self.poner_el_juego(tamano)
                    self.ver_tablero(tablero,turno)
                
                else:
                    print("\nGracias por jugar, suerte para la próxima! :)" if turno==1 else "\nGracias por jugar, mejoraré para ganarte la próxima! >:)")
            
            else:
                print("\ntiro de maquina:")
                tablero, turno=self.tiro_random(tablero,turno)
                self.ver_tablero(tablero,turno)
                if(self.condiciones_de_terminado(tablero)):
                    print("\nVICTORIA DE:", "O\n\n\nQuieres seguir?:(1=si, 0=no)" if turno==1 else "X\n\n\nQuieres seguir?:(1=si, 0=no)")
                    seguir=int(input())
                    if(seguir!=0):
                        tablero, turno=self.poner_el_juego(tamano)
                        self.ver_tablero(tablero,turno)
                    
                    else:
                        print("\nGracias por jugar, suerte para la próxima! :)" if turno==1 else "\nGracias por jugar, mejoraré para ganarte la próxima! >:)")
    
if __name__=="__main__":
    juego=hexapawn()

    juego.juego_random_contra_humano(3)

JUEGO CONTRA HUMANO:

╔═══════════╗   filas     turno de:  X
║ O │ O │ O ║  0
║---┼---┼---║
║   │   │   ║  1
║---┼---┼---║
║ X │ X │ X ║  2
╚═══════════╝
  0   1   2  

columnas



╔═══════════╗   filas     turno de:  O
║ O │ O │ O ║  0
║---┼---┼---║
║   │   │ X ║  1
║---┼---┼---║
║ X │ X │   ║  2
╚═══════════╝
  0   1   2  

columnas



tiro de maquina:

╔═══════════╗   filas     turno de:  X
║   │ O │ O ║  0
║---┼---┼---║
║ O │   │ X ║  1
║---┼---┼---║
║ X │ X │   ║  2
╚═══════════╝
  0   1   2  

columnas




ValueError: invalid literal for int() with base 10: ''

### DEFINICIÓN DE NODOS DEL ARBOL

In [48]:
class nodo:
    """
    Clase para representar los nodos del árbol.
    """

    def __init__(self, tablero, turno, g_p):
        #estructuras de navegación
        self.pad=None            # referencia al nodo padre
        self.hijos = []          # lista de referencias a los nodos hijos

        #datos contenida
        self.tablero = tablero   # tablero del juego específico en el nodo
        self.g_p = g_p           # 1: ganador, -1: perdedor, 0: sigue el juego
        self.uct = 6.0           # calificación asignada al tablero
        self.n = 0               # cantidad de veces que se ha visitado el nodo
        self.wins = 0            # cantidad de victorias del subárbol
        self.turno = turno       # turno del siguiente tiro

### ESTRUCTURA DEL ARBOL

In [49]:
class arbol:
    def __init__(self, clase_juego, seed=27779):
        self.seed=int(seed)
        self.gen=random.Random(self.seed)
        self.juego=clase_juego

    #Regresa un número entero aleatorio
    def random_int(self):
        return self.gen.randint(1, 1000)
    
    #Regresa un número real aleatorio entre 0 y 1
    def random_real(self):
        return self.gen.random()
    
    def insertar(self, padr, tablero_, turno_, g_p_):
        if(padr!=None):
            hijo=nodo(tablero_, turno_, g_p_)
            hijo.pad=padr
            padr.hijos.append(hijo)
            return hijo
        
        else:
            return None
        
    #RECORRIDOS
    def postorden(self, actual):
        if(actual!=None):
            for i in range(len(actual.hijos)):
                self.postorden(actual.hijos[i])

            print()

            for i in range(len(actual.tablero)):
                for j in range(len(actual.tablero[0])):
                    if(actual.tablero[i][j]==0):
                        print(" ", end="")
                    
                    elif(actual.tablero[i][j]==1):
                        print("X", end="")
                        
                    else:
                        print("O", end="")
                    
                print()
            
    def preorden(self, actual):
        if(actual!=None):
            print()
            for i in range(len(actual.tablero)):
                for j in range(len(actual.tablero[0])):
                    if(actual.tablero[i][j]==0):
                        print(" ", end="")
                    
                    elif(actual.tablero[i][j]==1):
                        print("X", end="")
                    
                    else:
                        print("O", end="")

                print()

            print("turno: ", actual.turno, "g_p: ", actual.g_p)

            for i in range(len(actual.hijos)):
                self.preorden(actual.hijos[i])
                
    def fancy_print(self, actual, indentacion=0):
        if(actual!=None):
            for i in range(len(actual.hijos)):
                self.fancy_print(actual.hijos[i], indentacion+7)

            for i in range(len(actual.tablero)):
                for k in range(indentacion):
                    print(" ", end="")

                for j in range(len(actual.tablero[0])):
                    if(actual.tablero[i][j]==0):
                        print(" ", end="")
                    elif(actual.tablero[i][j]==1):
                        print("X", end="")
                    else:
                        print("O", end="")

                print()

            for k in range(indentacion):
                print(" ", end="")

            print("trn ", actual.turno, " g/p ", actual.g_p, " uct ", actual.uct, " vsts ", actual.n, " wns ", actual.wins, "\n")

    def calcular_uct(self, nodo, constante=2):
        if(nodo!=None):
            if(nodo.pad!=None):
                if(nodo.n!=0):
                    nodo.uct=((nodo.wins*1.0)/(nodo.n*1.0))+np.sqrt((constante*np.log(nodo.pad.n))/(nodo.n*1.0))
                
                else:
                    nodo.uct=6.0
                    
            else:
                if(nodo.n!=0):
                    nodo.uct=((nodo.wins*1.0)/(nodo.n*1.0))
                
                else:
                    nodo.uct=6.0

        '''          
        la lógica de este paso es:
        si el nodo no es nulo:

            si el padre del nodo no es nulo:
                si visitas distinto de cero:
                    nodo->uct=formula completa
                demás:
                    nodo->uct=0
            demás: 
                si visitas distinto de cero:
                    nodo->uct=promedio ponderado (solamente wins entre visitas)
                demás:
                    nodo->uct=0
        '''

    def montecarlo_tree_search_random_search(self, tablero, turno, tamano, exploracion=2, verbose=False):
        tablero_copia=tablero.copy()
        turno_copia=turno

        if not self.juego.condiciones_de_terminado(tablero):
            raiz=nodo(tablero, turno, 0)
            nodo_desechable=None
            nodo_backprop=None
            nodos_cantidad=0
                        
            tiro, turno_siguiente=self.juego.tiro_random(tablero, turno)

            nodo_desechable=self.insertar(raiz, tiro, turno_siguiente, 0 if not self.juego.condiciones_de_terminado(tiro) else 2 if turno_siguiente==1 else 1)
            
            if(turno_copia==self.juego.terminar_juego(tiro, turno_siguiente)):
                nodo_desechable.wins+=1
                nodo_desechable.n+=1
                raiz.wins+=1
                raiz.n+=1
            else:
                nodo_desechable.n+=1
                raiz.n+=1
            
            self.calcular_uct(nodo_desechable, exploracion)
            self.calcular_uct(raiz, exploracion)
            nodo_desechable=None

            while(nodos_cantidad<tamano):
                nodo_desechable=raiz
                tablero=nodo_desechable.tablero
                turno=nodo_desechable.turno
                while nodo_desechable.hijos:
                    tiro, turno_siguiente=self.juego.tiro_random(tablero, turno)

                    for i in range(len(nodo_desechable.hijos)):
                        if(nodo_desechable.hijos[i].tablero==tiro):
                            nodo_desechable=nodo_desechable.hijos[i]
                            break
                        
                        if(i==nodo_desechable.hijos.size()-1):
                            i=-1
                            break
                        
                            
                    #SELECCION
                    if(i==-1):
                        if(turno_siguiente!=0):
                            #EXPANSION
                            nodo_desechable=self.insertar(nodo_desechable, tiro, turno_siguiente, 0 if not self.juego.condiciones_de_terminado(tiro) else 2 if turno_siguiente==1 else 1)
                            nodo_backprop=nodo_desechable

                            if(nodo_backprop.g_p==turno_copia):
                                resultado=1
                            
                            elif(nodo_backprop.g_p!=0):
                                resultado=0
                            
                            else:
                                resultado=1 if turno_copia==self.juego.terminar_juego(tiro, turno_siguiente) else 0
                            
                            nodo_backprop.n+=1
                            if(resultado==1):
                                nodo_backprop.wins+=1
                            

                            while(nodo_backprop.pad!=None):
                                nodo_backprop.pad.wins+=resultado
                                nodo_backprop.pad.n+=1
                                self.calcular_uct(nodo_backprop, exploracion)
                                nodo_backprop=nodo_backprop.pad
                            
                            self.calcular_uct(nodo_backprop, exploracion)
                        
                        else:
                            break
                        
                    
                    else:
                        #if no tiene hijos, darle uno, y regresar a la raiz
                        if(nodo_desechable.g_p!=0):
                            break
                        
                        elif not nodo_desechable.hijos:
                            tutiro, turno_siguientepla=self.juego.tiro_random(tiro, turno_siguiente);

                            #EXPANSION
                            nodo_backprop=self.insertar(nodo_desechable, tiro, turno_siguiente, 0 if not self.juego.condiciones_de_terminado(tiro) else 2 if turno_siguiente==1 else 1)

                            if(nodo_backprop.g_p==turno_copia):
                                resultado=1
                            
                            elif(nodo_backprop.g_p!=0):
                                resultado=0
                            
                            else:
                                resultado=1 if turno_copia==self.juego.terminar_juego(tiro, turno_siguiente) else 0
                            
                            nodo_backprop.n+=1
                            nodo_backprop.wins+=resultado

                            while(nodo_backprop.pad!=None):
                                nodo_backprop.pad.wins+=resultado
                                nodo_backprop.pad.n+=1
                                self.calcular_uct(nodo_backprop, exploracion)
                                nodo_backprop=nodo_backprop.pad
                            
                            self.calcular_uct(nodo_backprop, exploracion)
                            nodo_desechable=raiz
                            tablero=tablero_copia
                            turno=turno_copia
                            break
                        
                        else:
                            tablero=nodo_desechable.tablero
                            turno=self.juego.siguiente_turno(turno)


                nodos_cantidad+=1
            
            if(verbose):
                self.fancy_print(raiz)
            
            max_=0
            if(raiz.hijos.size()==1):
                tablero=raiz.hijos[0].tablero
            
            else:
                for i in range(len(raiz.hijos)):
                    if(raiz.hijos[i].uct>max_):
                        max_=raiz.hijos[i].uct
                        tablero=raiz.hijos[i].tablero
                        
            return tablero, 2 if turno_copia==1 else 1
        
        else:
            if(verbose):
                print("NO_POSSIBLE_SHOT")
            
            return tablero,turno

    def montecarlo_tree_search_probabilistic_search(self, tablero, turno, tamano, exploracion=2, verbose=False):

        tablero_copia=tablero.copy()

        turno_copia=turno
        
        if not self.juego.condiciones_de_terminado(tablero):
            raiz=nodo(tablero, turno, 0)
            nodo_desechable=None
            nodo_backprop=None
            nodos_cantidad=0
            max_=0.0
            
            #a la raiz, darle todos los hijos que pueda tener, y darles uct
            tableros, turnos=self.juego.todos_los_tiros(tablero,turno)

            for i in range(len(tableros)):
                nodo_backprop=self.insertar(raiz, 
                                            tableros[i], 
                                            turnos, 
                                            0 if not self.juego.condiciones_de_terminado(tableros[i]) else 2 if turnos==1 else 1)
                self.calcular_uct(nodo_backprop, exploracion)

            self.calcular_uct(raiz, exploracion)
            nodo_desechable=None
            while(nodos_cantidad<tamano):
                nodo_desechable=raiz
                #SELECCION
                while nodo_desechable.hijos:
                    for i in range(len(nodo_desechable.hijos)):
                        self.calcular_uct(nodo_desechable.hijos[i], exploracion)
                    
                    max_=0.0

                    for i in range(len(nodo_desechable.hijos)):
                        if(nodo_desechable.hijos[i].uct>max_):
                            max_=nodo_desechable.hijos[i].uct
                            j=i
                            
                    nodo_desechable=nodo_desechable.hijos[j]
                
                if(nodo_desechable!=None):
                    if not self.juego.condiciones_de_terminado(nodo_desechable.tablero):
                        #EXPANSION
                        tableros, turnos=self.juego.todos_los_tiros(nodo_desechable.tablero, 2 if nodo_desechable.turno==1 else 1)

                        if tableros:
                            for i in range(len(tableros)):
                                self.insertar(nodo_desechable, tableros[i], turnos, 0 if not self.juego.condiciones_de_terminado(tableros[i]) else 1 if turnos==1 else 2)
                            
                            nodo_backprop=nodo_desechable.hijos[len(nodo_desechable.hijos-1)]
                            #SIMULACION
                            if(nodo_backprop.g_p==turno_copia):
                                resultado=1
                            
                            elif(nodo_backprop.g_p!=0):
                                resultado=0
                            
                            else:
                                resultado=1 if turno_copia==self.juego.terminar_juego(nodo_backprop.tablero, turnos) else 0
                            
                            #BACKPROP
                            while(nodo_backprop.pad!=None):
                                nodo_backprop.pad.wins+=resultado
                                nodo_backprop.pad.n+=1
                                nodo_backprop=nodo_backprop.pad
                                
                    else:
                        if(nodo_desechable.g_p==turno_copia):
                            resultado=1
                        
                        else:
                            resultado=0
                        
                        #BACKPROP
                        nodo_backprop=nodo_desechable
                        nodo_backprop.n+=1
                        nodo_backprop.wins+=resultado
                        while(nodo_backprop.pad!=None):
                            nodo_backprop.pad.wins+=resultado
                            nodo_backprop.pad.n+=1
                            nodo_backprop=nodo_backprop.pad
                            
                nodos_cantidad+=1
            
            if(verbose):
                self.fancy_print(raiz)
            
            max_=0.0
            if len(raiz.hijos)==1:
                tablero=raiz.hijos[0].tablero
            
            else:
                for i in range(len(raiz.hijos)):
                    if(raiz.hijos[i].uct>max_):
                        max_=raiz.hijos[i].uct
                        tablero=raiz.hijos[i].tablero
                    
                
            return tablero, 2 if turno_copia==1 else 1
        
        else:
            if(verbose):
                print("NO_POSSIBLE_SHOT")
            
            return tablero,turno
        
    def leer_entero(self):
        while True:
            try:
                x = int(input())
                
                if x == 123456789:
                    sys.exit(2)
                
                return x

            except ValueError:
                print("Entrada inválida. Intenta de nuevo: ", end="")


    def leer_double(self):
        while True:
            try:
                x = float(input())
                
                if x == 123456789:
                    sys.exit(1)
                
                return x

            except ValueError:
                print("Entrada inválida. Intenta de nuevo: ", end="")

    def _juego_contra_humano_inteligente(self, tamano, difaul=0, primero=0, verbose=0, dificultad=-1, tipo_de_busqueda=True):
        print("JUEGO CONTRA HUMANO:")
        tablero, turno=self.juego.poner_el_juego(tamano, difaul)

        seguir=1
        victoria_humana=0
        victoria_maquina=0
        self.juego.ver_tablero(tablero, turno)

        while(seguir):
            if(primero):
                print("\ntiro de maquina:\n", end="")
                turno=self.juego.siguiente_turno(turno)
                if(tipo_de_busqueda):
                    tablero, turno=self.montecarlo_tree_search_probabilistic_search(tablero,
                                                                                    turno, 
                                                                                    50000 if dificultad==-1 else int(dificultad*(0.02*np.pow(93.57, len(tablero)))),
                                                                                    self.juego.siguiente_turno(turno), 
                                                                                    verbose)
                
                else:
                    tablero, turno=self.montecarlo_tree_search_random_search(tablero,
                                                                             turno, 
                                                                             50000 if dificultad==-1 else int(dificultad*(0.02*np.pow(93.57, len(tablero)))), 
                                                                             self.juego.siguiente_turno(turno), 
                                                                             verbose)

                self.juego.ver_tablero(tablero,turno)
            
            while(seguir):
                while(1):
                    print("\n\n\nFilas(origen):", end="")
                    f_o=self.leer_entero()
                    print("\nColumnas(origen):", end="")
                    c_o=self.leer_entero()
                    print("\nFilas(terminal):", end="")
                    f_t=self.leer_entero()
                    print("\nColumnas(terminal):", end="")
                    c_t=self.leer_entero()

                    if(self.juego.tiro_legal(f_o,c_o,f_t,c_t,tablero) and (tablero[f_o][c_o]==1 if difaul==0 else difaul)):
                        tablero[f_o][c_o]=0
                        tablero[f_t][c_t]=1 if difaul==0 else difaul
                        break
                    
                    else:
                        print("\nTiro no permitido, intenta de nuevo.", end="")
                    
                
                turno=self.juego.siguiente_turno(turno)
                self.juego.ver_tablero(tablero,turno)
                if(self.juego.condiciones_de_terminado(tablero)):
                    print("\nVICTORIA DE:", end="")
                    if(turno==1):
                        if difaul==0:
                            victoria_maquina+=1
                        else:
                            if difaul==1:
                                victoria_maquina+=1
                            else:
                                victoria_humana+=1
                        
                        print("O\n\n-------------------------------MARCADOR------------------------------\n", end="")
                        print("╔═══════════════════════════════════════════════════════════════════╗\n", end="")
                        print("║ Humano:      ", victoria_humana, "                                                  ║\n", end="")
                        if(victoria_humana>victoria_maquina):
                            print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: HUMANO ║\n", end="")
                        
                        elif(victoria_humana<victoria_maquina):
                            print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: MAQUINA║\n", end="")
                        
                        else:
                            print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: TABLAS ║\n", end="")
                        
                        print("║ Computadora: ", victoria_maquina, "                                                  ║\n", end="")
                        print("╚═══════════════════════════════════════════════════════════════════╝\n", end="")
                        print("\n\n\nQuieres seguir?:(1=si, 0=no)", end="")
                    
                    else:
                        if difaul==0:
                            victoria_humana+=1
                        else:
                            if difaul==1:
                                victoria_humana+=1
                            else:
                                victoria_maquina+=1

                        print("X\n\n-------------------------------MARCADOR------------------------------\n", end="")
                        print("╔═══════════════════════════════════════════════════════════════════╗\n", end="")
                        print("║ Humano:      ", victoria_humana, "                                                  ║\n", end="")
                        if(victoria_humana>victoria_maquina):
                            print("║                             Razón:"<< fixed << setprecision(2)<<(victoria_humana*1.0)/(victoria_maquina*1.0)<<"            Ventaja: HUMANO ║\n", end="")
                        
                        elif(victoria_humana<victoria_maquina):
                            print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: MAQUINA║\n", end="")
                        
                        else:
                            print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: TABLAS ║\n", end="")
                        
                        print("║ Computadora: ", victoria_maquina, "                                                  ║\n", end="")
                        print("╚═══════════════════════════════════════════════════════════════════╝\n", end="")
                        print("\n\n\nQuieres seguir?:(1=si, 0=no)", end="")
                    
                    seguir=int(input())
                    if(seguir==1):
                        tablero, turno=self.juego.poner_el_juego(tamano)
                        self.juego.ver_tablero(tablero,turno)
                        break
                    
                    else:
                        if(turno==1):
                            print("\nGracias por jugar, suerte para la próxima! :)", end="")
                        
                        else:
                            print("\nGracias por jugar, mejoraré para ganarte la próxima! >:)", end="")
                        
                    
                
                else:
                    print("\ntiro de maquina:\n", end="")
                    if(dificultad!=-1):
                        print("nodos a calcular: ", int(dificultad*(0.02*pow(93.57,tablero.size()))))
                    
                    else:
                        print("nodos a calcular: 50000")
                    
                    if(tipo_de_busqueda):
                        tablero, turno=self.montecarlo_tree_search_probabilistic_search(tablero,
                                                                                        turno, 
                                                                                        50000 if dificultad==-1 else int(dificultad*(0.02*np.pow(93.57, len(tablero)))), 
                                                                                        self.juego.siguiente_turno(turno), 
                                                                                        verbose)
                    
                    else:
                        tablero, turno=self.montecarlo_tree_search_random_search(tablero,
                                                                             turno, 
                                                                             50000 if dificultad==-1 else int(dificultad*(0.02*np.pow(93.57, len(tablero)))), 
                                                                             self.juego.siguiente_turno(turno), 
                                                                             verbose)
                    
                    self.juego.ver_tablero(tablero,turno)
                    if(self.juego.condiciones_de_terminado(tablero)):
                        print("\nVICTORIA DE:", end="")
                        if(turno==1):
                            if difaul==0:
                                victoria_maquina+=1
                            else:
                                if difaul==1:
                                    victoria_maquina+=1
                                else:
                                    victoria_humana+=1
                            print("O\n\n-------------------------------MARCADOR------------------------------\n", end="")
                            print("╔═══════════════════════════════════════════════════════════════════╗\n", end="")
                            print("║ Humano:      ", victoria_humana, "                                                  ║\n", end="")
                            if(victoria_humana>victoria_maquina):
                                print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: HUMANO ║\n", end="")
                            
                            elif(victoria_humana<victoria_maquina):
                                print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: MAQUINA║\n", end="")
                            
                            else:
                                print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: TABLAS ║\n", end="")
                            
                            print("║ Computadora: ", victoria_maquina, "                                                  ║\n", end="")
                            print("╚═══════════════════════════════════════════════════════════════════╝\n", end="")
                            print("\n\n\nQuieres seguir?:(1=si, 0=no)", end="")

                        else:
                            if difaul==0:
                                victoria_humana+=1
                            else:
                                if difaul==1:
                                    victoria_humana+=1
                                else:
                                    victoria_maquina+=1
                            print("X\n\n-------------------------------MARCADOR------------------------------\n", end="")
                            print("╔═══════════════════════════════════════════════════════════════════╗\n", end="")
                            print("║ Humano:      ", victoria_humana, "                                                  ║\n", end="")
                            if(victoria_humana>victoria_maquina):
                                print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: HUMANO ║\n", end="")
                            
                            elif(victoria_humana<victoria_maquina):
                                print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: MAQUINA║\n", end="")
                            
                            else:
                                print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: TABLAS ║\n", end="")
                            
                            print("║ Computadora: ", victoria_maquina, "                                                  ║\n", end="")
                            print("╚═══════════════════════════════════════════════════════════════════╝\n", end="")
                            print("\n\n\nQuieres seguir?:(1=si, 0=no)", end="")
                        
                        seguir=int(input())
                        if(seguir):
                            tablero, turno=self.juego.poner_el_juego(tamano)
                            self.juego.ver_tablero(tablero,turno);
                            break
                        
                        else:
                            if(turno==1):
                                print("\nGracias por jugar, suerte para la próxima! :)\n", end="")
                            
                            else:
                                print("\nGracias por jugar, mejoraré para ganarte la próxima! >:)\n", end="")


        print("----------------------------MARCADOR FINAL---------------------------\n", end="")
        print("╔═══════════════════════════════════════════════════════════════════╗\n", end="")
        print("║ Humano:      ", victoria_humana, "                                                  ║\n", end="")
        if(victoria_humana>victoria_maquina):
            print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: HUMANO ║\n", end="")
        
        elif(victoria_humana<victoria_maquina):
            print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: MAQUINA║\n", end="")
        
        else:
            print("║                             Razón:", (victoria_humana*1.0)/(victoria_maquina*1.0), "            Ventaja: TABLAS ║\n", end="")
        
        print("║ Computadora: ", victoria_maquina, "                                                  ║\n", end="")
        print("╚═══════════════════════════════════════════════════════════════════╝\n", end="")
        if(victoria_humana==victoria_maquina):
            print("-EMPATE\n\n", end="")
        
        elif(victoria_humana>victoria_maquina):
            print("-GANASTE\n\n", end="")
        
        else:
            print("-PERDISTE\n\n", end="")

    def juego_contra_humano_inteligente(self):
        print("\n\nQuieres el juego default? (1=si, 0=entrar a configuracion)")
        setinds=self.leer_entero()
        if(setinds==0):
            print("\n\nQué piezas quieres tener? (X=1, O=2)  ")
            piezas=self.leer_entero()
            print("\n\nQuién quieres que inicie primero? (tu=0, computadora=1) ")
            primero=True if self.leer_entero()!=0 else False
            print("\n\nQué tamaño quieres que tenga el tablero?")
            tamanio=self.leer_entero()
            tamanio=3 if tamanio<3 else tamanio
            print("\n\nQuieres ver el arbol de probabilidad? (1=si, 0=no) (RECOMENDABLE NO VERLO, PUEDEN SER ARBOLES MUY GRANDES)")
            ver_=True if self.leer_entero()!=0 else False
            print("\n\nQué dificultad quieres para el juego? (valor entre 0 y 1, 0 es muy facil, 1 es muy dificil)")
            dificultad=self.leer_double()
            print("\n\nQué tipo de búsqueda quieres que se haga? (random=0, probabilistica=1) ")
            busq=True if self.leer_entero()!=0 else False
            self._juego_contra_humano_inteligente(tamanio, piezas, primero, ver_, dificultad, busq)
        
        else:
            self._juego_contra_humano_inteligente(3, 1, False, False, -1, True)

if __name__=="__main__":
    clasi=arbol(hexapawn())
    clasi.juego_contra_humano_inteligente()



Quieres el juego default? (1=si, 0=entrar a configuracion)
JUEGO CONTRA HUMANO:

╔═══════════╗   filas     turno de:  X
║ O │ O │ O ║  0
║---┼---┼---║
║   │   │   ║  1
║---┼---┼---║
║ X │ X │ X ║  2
╚═══════════╝
  0   1   2  

columnas





Filas(origen):
Columnas(origen):
Filas(terminal):
Columnas(terminal):
╔═══════════╗   filas     turno de:  X
║ O │ O │ O ║  0
║---┼---┼---║
║   │   │ X ║  1
║---┼---┼---║
║ X │ X │   ║  2
╚═══════════╝
  0   1   2  

columnas



tiro de maquina:
nodos a calcular: 50000

╔═══════════╗   filas     turno de:  O
║ O │ X │ O ║  0
║---┼---┼---║
║ X │ X │   ║  1
║---┼---┼---║
║   │   │   ║  2
╚═══════════╝
  0   1   2  

columnas



VICTORIA DE:X

-------------------------------MARCADOR------------------------------
╔═══════════════════════════════════════════════════════════════════╗
║ Humano:       1                                                   ║


ZeroDivisionError: float division by zero